# Notebook: BCI_40_CarDet_Evalua_Salida
*********************************************************************************

## Informacion del Notebook

### Encabezado
**************************************************************************
* Nombre: BCI_40_CarDet_Evalua_Salida.ipynb
* Ruta: https://adb-5512273708018582.2.azuredatabricks.net/?o=5512273708018582#notebook/2997520011900276
* Autor: Gabriel MartÍnez (SimpleData) - Ing. SW BCI: Jonatan Cancino
* Fecha: 12/08/2022
* Descripcion: Evaluacion criterios de salida de deterioro.
* Documentacion:
***************************************************************************

### Mantenciones
**************************************************************************
#### Mantención Nro: 1
* Autor: Gagriel Martinez (SimpleData) - Ing. SW BCI: Jonatan Cancino
* Fecha: 10/02/2025 
* Descripción: Se agregan los Clientes que salen por LIR     
***************************************************************************

**************************************************************************
#### Mantención Nro: 2
* Autor: Gabriel Martinez (SimpleData) - Ing. SW BCI: Jonatan Cancino
* Fecha: 09/04/2025 
* Descripción: Se eliminan las reglas de evaluacion para los Clientes que salen por Criterio LIR.     
***************************************************************************

**************************************************************************
#### Mantención Nro: 3
* Autor: Gabriel Martinez (SimpleData) - Ing. SW BCI: Jonatan Cancino
* Fecha: 22/07/2025 
* Descripción: Para la evaluacion si el cliente u operacion sin refinanciamiento ni curse bajo mora, se remplaza la tabla tbl_cd_ope_condicion_ren por tbl_cd_ope_condicion_sal_ren.
***************************************************************************

### Tablas Entrada y Salida
**************************************************************************
#### Tablas Entrada: 
* {base_silver_x}.tbl_cd_cartdet_crit_ent_prin
* {base_silver_x}.tbl_cd_d00_segmentado
* {base_silver_x}.tbl_cd_segmentacion_cliente
* {base_silver_x}.tbl_cd_cliente_consolidado
* {base_silver_x}.tbl_cd_cliente_det_ssff 
* {base_silver_x}.tbl_cd_cliente_det_fact
* {base_silver_x}.tbl_cd_cliente_lir
* {base_silver_x}.tbl_cd_ope_dia_mora
* {base_silver_x}.tbl_cd_ope_pag_cons_ibm
* {base_silver_x}.tbl_cd_dat_ope_ini_cd
* {base_silver_x}.tbl_cd_ope_condicion_ren
* {base_silver_x}.tbl_cd_ope_curse_bajo_mora
* {base_silver_x}.tbl_cd_ope_ent_meses
***************************************************************************
#### Tablas Salida: 
* {base_silver_x}.tbl_cd_cartdet_crit_sal_ope_eval
***************************************************************************


## Carga Dependencias

### Carga funciones comunes

In [0]:
%run "./Funciones_Comunes"

# Notebook: Funciones_Comunes
**************************************************************************

## Informacion del Notebook 

### Encabezado
**************************************************************************
* Nombre: Funciones_Comunes.ipynb
* Ruta: https://adb-5512273708018582.2.azuredatabricks.net/?o=5512273708018582#notebook/3570959530595695
* Autor: Gabriel Martínez (SimpleData) - Ing. SW BCI: Jonatan Cancino
* Fecha: 23/09/2023
* Descripcion: Notebook con funciones genéricas que pueden ser usadas por otros notebooks.
* Documentacion:
***************************************************************************

### Mantenciones
**************************************************************************
#### Mantención Nro: 1
* Autor: Gabriel Martinez (SimpleData) - Ing. SW BCI: Jonatan Cancino
* Fecha: 15/02/2025 
* Descripción: Se cambio el metodo de cancelacion utilizando el comando (raise) y se incorporada la funcion de ir a buscar el ultimo dia calendario. Tambien se agrego una nueva funcion (obtener_estados_tablas).  
***************************************************************************
#### Mantención Nro: 2
* Autor: Gabriel Martinez (SimpleData) - Ing. SW BCI: Jonatan Cancino
* Fecha: 25/04/2025 
* Descripción: Se modifico la funcion extension_archivos para que cuando la vigencia sea previa, asigne extencion .PRV.  
***************************************************************************
#### Mantención Nro: 3
* Autor: Gabriel Martinez (SimpleData) - Ing. SW BCI: Claudia Yañez
* Fecha: 08/07/2025 
* Descripción: Se realiza una mejora en la funcion mostrar_variacion_criterio.  
***************************************************************************

## Carga librerias

...

### obtener archivo periodo anterior

### concatena archivos

###primer_dia_mes_sig

###Calcula fecha X meses atras

## FIN definición de funciones

## Parámetros

### Setea Parámetros

In [0]:

dbutils.widgets.text("fecha_w","","01-Fecha:")
dbutils.widgets.text("bd_silver_w","","03-Nombre BD Silver:")

fecha_x = dbutils.widgets.get("fecha_w")
base_silver_x = dbutils.widgets.get("bd_silver_w")

spark.conf.set("bci.fecha", fecha_x)
spark.conf.set("bci.dbnamesilver", base_silver_x)

print(f"Fecha de Proceso actual: [fecha_x] {fecha_x}")
print(f"Nombre BD Silver: [base_silver_x] {base_silver_x}")


Fecha de Proceso actual: [fecha_x] 20250930
Nombre BD Silver: [base_silver_x] dsr_gld_bciwork_db


### Valida parámetros

In [0]:
valida_parametro(fecha_x)

In [0]:
valida_parametro(base_silver_x)

## INICIO Proceso extraccion y transformacion
--------------------------------------
- Por cada fuente que se utilice se debe:
     - Titulo: generar un titulo generico, con nombre fuente, y descripcion del proposito de la extraccion
     - Extraer: para el periodo, o rango de fecha que se necesita la iformacion. Debe tener el prefijo tmp_EXT_{nombrefuente}
     - Transformar: generar la informacion necesaria para la salida final. Se pueden generar mas de una tabla temporal para llegar al resultado final. Debe tener el prefijo tmp_RES_{nombre}_correlativo


### Parametrizacion
---
* define y asigna valores a los parametros


In [0]:
#parametros internos notebook
p_cod_seg_ind='I'
p_evaluaciones = 'S01','S02','S03','S04','S05','S06','S07','S08','S09','S10','S11','S12','S13','S14'

print(f"p_cod_seg_ind: {p_cod_seg_ind}")
print(f"p_evaluaciones: {p_evaluaciones}")


p_cod_seg_ind: I
p_evaluaciones: ('S01', 'S02', 'S03', 'S04', 'S05', 'S06', 'S07', 'S08', 'S09', 'S10', 'S11', 'S12', 'S13', 'S14')


In [0]:
resultado = obtener_parametros_de_tbl(base_silver_x)
param_cal = resultado[0]
p_fec_cv = resultado[1]
p_crit_det_sal_tot_ifrs = resultado[2]
p1_valordvcn = resultado[3]
p1_valordvcx = resultado[4]
p1_valorcdir = resultado[5]
p1_valordmor = resultado[6]
p_cantminmes = resultado[7]
p1_diasmora = resultado[8]
p_pagcons = resultado[9]
p_cant_min_pag = resultado[10]
p3_montosbif = resultado[11]
p1_valor90180 = resultado[17]
p1_valor180 = resultado[18]
p1_valor3090 = resultado[19]


print('Valor param_cal :',param_cal )
print('Valor p_fec_cv :',p_fec_cv )
print('Valor p_crit_det_sal_tot_ifrs :',p_crit_det_sal_tot_ifrs)
print('Valor p1_valordvcn :',p1_valordvcn)
print('Valor p1_valordvcx :',p1_valordvcx)
print('Valor p1_valorcdir :',p1_valorcdir)
print('Valor p1_valordmor :',p1_valordmor)
print('Valor p_cantminmes :',p_cantminmes)
print('Valor p1_diasmora :',p1_diasmora)
print('Valor p_pagcons :',p_pagcons)
print('Valor p_cant_min_pag :',p_cant_min_pag)
print('Valor p3_montosbif :',p3_montosbif)
print('Valor p1_valor90180 :',p1_valor90180)
print('Valor p1_valor180 :',p1_valor180)
print('Valor p1_valor3090 :',p1_valor3090)


Valor param_cal : ('9', '10', '11', '12', '13', '14', '15', '16')
Valor p_fec_cv : 20061201
Valor p_crit_det_sal_tot_ifrs : 0
Valor p1_valordvcn : 1
Valor p1_valordvcx : 1
Valor p1_valorcdir : 1
Valor p1_valordmor : 1
Valor p_cantminmes : 4
Valor p1_diasmora : 30
Valor p_pagcons : 4
Valor p_cant_min_pag : 2
Valor p3_montosbif : 0
Valor p1_valor90180 : 1
Valor p1_valor180 : 1
Valor p1_valor3090 : 1


In [0]:
#Para el cálculo de clientes LIR se debe tomar los ultimos 12 meses a partir de la fecha de proceso
p_fec_12_meses_atras=fecha_x_meses_atras(fecha_x,12)
print(f"p_fec_12_meses_atras: {p_fec_12_meses_atras}")

p_fec_12_meses_atras: 20240930


### Extrae Operaciones Deteriorados
--------------------------------------
- Extrae todos las operaciones deterioradas del proceso actual
- Una operacion puede estar deteriorada por varios motivos, aca solo interesa la operacion deteriorada, no el motivo. 

In [0]:
paso_query1 = f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_EXT_tbl_cartdet_crit_ent_crit AS
SELECT  
  A.periodo_cierre             AS     periodo_cierre,
  A.fecha_cierre               AS     fecha_cierre,
  A.tipo_proceso               AS     tipo_proceso,
  A.rut_cliente                AS     rut_cliente,
  A.dv_rut_cliente             AS     dv_rut_cliente,
  A.tipo_operacion             AS     tipo_operacion,
  A.operacion                  AS     operacion,
  A.sistema                    AS     sistema,
  A.segmento                   AS     segmento,
  A.grupo                      AS     grupo,
  A.periodo_evaluacion         AS     periodo_evaluacion  
FROM 
    {base_silver_x}.tbl_cd_cartdet_crit_ent_prin A
WHERE
    A.fecha_cierre =   {fecha_x} 
QUALIFY  ROW_NUMBER() OVER(PARTITION BY A.operacion, sistema ORDER BY A.fecha_cierre DESC, A.periodo_evaluacion ASC) =1
"""

In [0]:
sql_safe(paso_query1)

sql_safe: query -> 
CREATE OR REPLACE TEMPORARY VIEW tmp_EXT_tbl_cartdet_crit_ent_crit AS
SELECT  
  A.periodo_cierre             AS     periodo_cierre,
  A.fecha_cierre               AS     fecha_cierre,
  A.tipo_proceso               AS     tipo_proceso,
  A.rut_cliente                AS     rut_cliente,
  A.dv_rut_cliente             AS     dv_rut_cliente,
  A.tipo_operacion             AS     tipo_operacion,
  A.operacion                  AS     operacion,
  A.sistema                    AS     sistema,
  A.segmento                   AS     segmento,
  A.grupo                      AS     grupo,
  A.periodo_evaluacion         AS     periodo_evaluacion  
FROM 
    dsr_gld_bciwork_db.tbl_cd_cartdet_crit_ent_prin A
WHERE
    A.fecha_cierre =   20250930 
QUALIFY  ROW_NUMBER() OVER(PARTITION BY A.operacion, sistema ORDER BY A.fecha_cierre DESC, A.periodo_evaluacion ASC) =1



DataFrame[]

### Evaluacion: cliente individual o grupal
---
* Si cliente es individual marca todas las operaciones con flag =1 


In [0]:
paso_query100 =  f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CAMPO_EVAL_1 as
SELECT
 A.periodo_cierre                              AS periodo_cierre,
 A.fecha_cierre                                AS fecha_cierre,
 A.tipo_proceso                                AS tipo_proceso,
 A.segmento                                    AS segmento,
 A.operacion                                   AS operacion,
 A.tipo_operacion                              AS tipo_operacion,
 A.sistema                                     AS sistema,
 A.rut_cliente                                 AS rut_cliente,
 A.dv_rut_cliente                              AS dv_rut_cliente ,
 'cli_segmento_cliente'                            AS nombre_campo,
 cast(A.cli_segmento_cliente as string)            AS valor_campo,
 'Cliente con segmento individual'                 AS condicion_regla,     
 "[eq to {p_cod_seg_ind}]"                               AS valor_regla,
 CASE 
     WHEN IFNULL(trim(A.cli_segmento_cliente),'SD') = '{p_cod_seg_ind}' 
     THEN 1 
     ELSE 0 
 END                                          AS flag_resultado_regla,
 'S01'                                        AS cod_evaluacion
FROM 
    {base_silver_x}.tbl_cd_ope_condicion_salida_deterioro  A
"""


In [0]:
sql_safe(paso_query100)

sql_safe: query -> 
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CAMPO_EVAL_1 as
SELECT
 A.periodo_cierre                              AS periodo_cierre,
 A.fecha_cierre                                AS fecha_cierre,
 A.tipo_proceso                                AS tipo_proceso,
 A.segmento                                    AS segmento,
 A.operacion                                   AS operacion,
 A.tipo_operacion                              AS tipo_operacion,
 A.sistema                                     AS sistema,
 A.rut_cliente                                 AS rut_cliente,
 A.dv_rut_cliente                              AS dv_rut_cliente ,
 'cli_segmento_cliente'                            AS nombre_campo,
 cast(A.cli_segmento_cliente as string)            AS valor_campo,
 'Cliente con segmento individual'                 AS condicion_regla,     
 "[eq to I]"                               AS valor_regla,
 CASE 
     WHEN IFNULL(trim(A.cli_segmento_cliente),'SD') = 'I' 
  

DataFrame[]

### Evaluacion: cliente SIN clasificacion de deterioro (2)
---
* Si cliente NO tiene clasificacion de deterioro entonces flag =1 


In [0]:
paso_query105 =  f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CAMPO_EVAL_2 as
SELECT
 A.periodo_cierre                                AS periodo_cierre,
 A.fecha_cierre                                  AS fecha_cierre,
 A.tipo_proceso                                  AS tipo_proceso,
 A.segmento                                      AS segmento,
 A.operacion                                     AS operacion,
 A.tipo_operacion                                AS tipo_operacion,
 A.sistema                                       AS sistema,
 A.rut_cliente                                   AS rut_cliente,
 A.dv_rut_cliente                                AS dv_rut_cliente ,
'cli_calificacion_bci'                               AS nombre_campo,
 cast(A.cli_calificacion_bci as string)              AS valor_campo,
 'Cliente sin calificacion de deterioro'         AS condicion_regla,     
 "[NOT IN {param_cal}]"                            AS valor_regla,
 CASE 
    WHEN trim(IFNULL(A.cli_calificacion_bci,'0')) NOT IN {param_cal} 
    THEN 1 
    ELSE 0 
  END                                            AS flag_resultado_regla,
 'S02'                                           AS cod_evaluacion  
FROM 
     {base_silver_x}.tbl_cd_ope_condicion_salida_deterioro A 
"""


In [0]:
sql_safe(paso_query105)

sql_safe: query -> 
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CAMPO_EVAL_2 as
SELECT
 A.periodo_cierre                                AS periodo_cierre,
 A.fecha_cierre                                  AS fecha_cierre,
 A.tipo_proceso                                  AS tipo_proceso,
 A.segmento                                      AS segmento,
 A.operacion                                     AS operacion,
 A.tipo_operacion                                AS tipo_operacion,
 A.sistema                                       AS sistema,
 A.rut_cliente                                   AS rut_cliente,
 A.dv_rut_cliente                                AS dv_rut_cliente ,
'cli_calificacion_bci'                               AS nombre_campo,
 cast(A.cli_calificacion_bci as string)              AS valor_campo,
 'Cliente sin calificacion de deterioro'         AS condicion_regla,     
 "[NOT IN ('9', '10', '11', '12', '13', '14', '15', '16')]"                            AS valor_regla,
 CAS

DataFrame[]

### Evaluacion: operaciones con saldo cero (27)
---
* Si operacion tiene saldo total ifrs cero entonces  flag =1 


In [0]:
paso_query110 =  f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CAMPO_EVAL_3 as
SELECT    
    A.periodo_cierre                                AS periodo_cierre,
    A.fecha_cierre                                  AS fecha_cierre,
    A.tipo_proceso                                  AS tipo_proceso,
    A.segmento                                      AS segmento,
    A.operacion                                     AS operacion,
    A.tipo_operacion                                AS tipo_operacion,
    A.sistema                                       AS sistema,
    A.rut_cliente                                   AS rut_cliente,
    A.dv_rut_cliente                                AS dv_rut_cliente ,
    'saldo_total_ifrs'                                              AS nombre_campo,
    cast(IFNULL(A.saldo_total_ifrs,0) as string)                     AS valor_campo,
    'Operacion con saldo igual o menor a  {p_crit_det_sal_tot_ifrs}'     AS condicion_regla,     
    '[<= {p_crit_det_sal_tot_ifrs}]'                       AS valor_regla,            
    CASE 
       WHEN IFNULL(A.saldo_total_ifrs,0) <= {p_crit_det_sal_tot_ifrs} 
       THEN 1 
       ELSE 0 
     END                                             AS flag_resultado_regla,
     'S03'                                           AS cod_evaluacion  
FROM 
    {base_silver_x}.tbl_cd_ope_condicion_salida_deterioro A
"""

In [0]:
sql_safe(paso_query110)

sql_safe: query -> 
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CAMPO_EVAL_3 as
SELECT    
    A.periodo_cierre                                AS periodo_cierre,
    A.fecha_cierre                                  AS fecha_cierre,
    A.tipo_proceso                                  AS tipo_proceso,
    A.segmento                                      AS segmento,
    A.operacion                                     AS operacion,
    A.tipo_operacion                                AS tipo_operacion,
    A.sistema                                       AS sistema,
    A.rut_cliente                                   AS rut_cliente,
    A.dv_rut_cliente                                AS dv_rut_cliente ,
    'saldo_total_ifrs'                                              AS nombre_campo,
    cast(IFNULL(A.saldo_total_ifrs,0) as string)                     AS valor_campo,
    'Operacion con saldo igual o menor a  0'     AS condicion_regla,     
    '[<= 0]'                       AS valor_r

DataFrame[]

### Evaluacion: Mora SBIF cliente (62)
---
* Si cliente tiene mora menor o igual al parametro entonces  flag =1 


In [0]:
paso_query115 =  f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CAMPO_EVAL_4 as
SELECT
    A.periodo_cierre                                            AS periodo_cierre,
    A.fecha_cierre                                              AS fecha_cierre,
    A.tipo_proceso                                              AS tipo_proceso,
    A.segmento                                                  AS segmento,
    A.operacion                                                 AS operacion,
    A.tipo_operacion                                            AS tipo_operacion,
    A.sistema                                                   AS sistema,
    A.rut_cliente                                               AS rut_cliente,
    A.dv_rut_cliente                                            AS dv_rut_cliente ,
    "cli_mora_sbif_f"                                           AS nombre_campo,
    cast(A.cli_mora_sbif_f as string)                             AS valor_campo,
    "Cliente con deuda morosa sbif menor o igual a: {p3_montosbif}"         AS condicion_regla,
    "[<= {p3_montosbif}]" AS valor_regla,           
    CASE 
         WHEN IFNULL(A.cli_mora_sbif_f,0) <= {p3_montosbif} 
         THEN 1 
         ELSE 0 
    END                                                         AS flag_resultado_regla,    
   'S04'                                                        AS cod_evaluacion  
FROM
    {base_silver_x}.tbl_cd_ope_condicion_salida_deterioro A 
"""

In [0]:
sql_safe(paso_query115)

sql_safe: query -> 
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CAMPO_EVAL_4 as
SELECT
    A.periodo_cierre                                            AS periodo_cierre,
    A.fecha_cierre                                              AS fecha_cierre,
    A.tipo_proceso                                              AS tipo_proceso,
    A.segmento                                                  AS segmento,
    A.operacion                                                 AS operacion,
    A.tipo_operacion                                            AS tipo_operacion,
    A.sistema                                                   AS sistema,
    A.rut_cliente                                               AS rut_cliente,
    A.dv_rut_cliente                                            AS dv_rut_cliente ,
    "cli_mora_sbif_f"                                           AS nombre_campo,
    cast(A.cli_mora_sbif_f as string)                             AS valor_campo,
    "Cliente con deuda

DataFrame[]

### Evaluacion: Clientes deteriorados SSFF (40)
---
* Si cliente NO esta informado deteriorado en SSFF entonces  flag =1 


In [0]:
paso_query120 =  f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CAMPO_EVAL_5 as
SELECT
    A.periodo_cierre                                            AS periodo_cierre,
    A.fecha_cierre                                              AS fecha_cierre,
    A.tipo_proceso                                              AS tipo_proceso,
    A.segmento                                                  AS segmento,
    A.operacion                                                 AS operacion,
    A.tipo_operacion                                            AS tipo_operacion,
    A.sistema                                                   AS sistema,
    A.rut_cliente                                               AS rut_cliente,
    A.dv_rut_cliente                                            AS dv_rut_cliente ,
    "flag_existe_cliente_ssff "                                 AS nombre_campo,
    cast(A.flag_existe_cliente_ssff as string)                  AS valor_campo,
    "Cliente sin deterioro SSFF"                                AS condicion_regla,
    "[eq to 0 (NOEXISTE)]"                                        AS valor_regla,           
    CASE 
         WHEN IFNULL(A.flag_existe_cliente_ssff,0) = 0 
         THEN 1 
         ELSE 0 
    END                                                         AS flag_resultado_regla,    
   'S05'                                                        AS cod_evaluacion  
FROM
     {base_silver_x}.tbl_cd_ope_condicion_salida_deterioro A 
"""

In [0]:
sql_safe(paso_query120)

sql_safe: query -> 
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CAMPO_EVAL_5 as
SELECT
    A.periodo_cierre                                            AS periodo_cierre,
    A.fecha_cierre                                              AS fecha_cierre,
    A.tipo_proceso                                              AS tipo_proceso,
    A.segmento                                                  AS segmento,
    A.operacion                                                 AS operacion,
    A.tipo_operacion                                            AS tipo_operacion,
    A.sistema                                                   AS sistema,
    A.rut_cliente                                               AS rut_cliente,
    A.dv_rut_cliente                                            AS dv_rut_cliente ,
    "flag_existe_cliente_ssff "                                 AS nombre_campo,
    cast(A.flag_existe_cliente_ssff as string)                  AS valor_campo,
    "Cliente sin deterio

DataFrame[]

### Evaluacion: Clientes deteriorados Factoring (41)
---
* Si cliente NO esta informado deteriorado en factoring entonces  flag =1 


In [0]:
paso_query125 =  f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CAMPO_EVAL_6 as
SELECT
    A.periodo_cierre                                            AS periodo_cierre,
    A.fecha_cierre                                              AS fecha_cierre,
    A.tipo_proceso                                              AS tipo_proceso,
    A.segmento                                                  AS segmento,
    A.operacion                                                 AS operacion,
    A.tipo_operacion                                            AS tipo_operacion,
    A.sistema                                                   AS sistema,
    A.rut_cliente                                               AS rut_cliente,
    A.dv_rut_cliente                                            AS dv_rut_cliente ,
    "flag_existe_cliente_fact "                                 AS nombre_campo,
    cast(A.flag_existe_cliente_fact as string)                  AS valor_campo,
    "Cliente sin deterioro FACTORING"                           AS condicion_regla,
    "[eq to 0 (NOEXISTE)]"                                       AS valor_regla,           
    CASE 
         WHEN IFNULL(A.flag_existe_cliente_fact,0) = 0 
         THEN 1 
         ELSE 0 
    END                                                         AS flag_resultado_regla,    
   'S06'                                                        AS cod_evaluacion  
FROM
     {base_silver_x}.tbl_cd_ope_condicion_salida_deterioro A 
"""

In [0]:
sql_safe(paso_query125)

sql_safe: query -> 
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CAMPO_EVAL_6 as
SELECT
    A.periodo_cierre                                            AS periodo_cierre,
    A.fecha_cierre                                              AS fecha_cierre,
    A.tipo_proceso                                              AS tipo_proceso,
    A.segmento                                                  AS segmento,
    A.operacion                                                 AS operacion,
    A.tipo_operacion                                            AS tipo_operacion,
    A.sistema                                                   AS sistema,
    A.rut_cliente                                               AS rut_cliente,
    A.dv_rut_cliente                                            AS dv_rut_cliente ,
    "flag_existe_cliente_fact "                                 AS nombre_campo,
    cast(A.flag_existe_cliente_fact as string)                  AS valor_campo,
    "Cliente sin deterio

DataFrame[]

### Evaluacion: clientes LIR (67)
---
* Si cliente NO esta informado en archivo LIR entonces  flag =1 


In [0]:
paso_query135 =  f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CAMPO_EVAL_7 as
SELECT
    A.periodo_cierre                                            AS periodo_cierre,
    A.fecha_cierre                                              AS fecha_cierre,
    A.tipo_proceso                                              AS tipo_proceso,
    A.segmento                                                  AS segmento,
    A.operacion                                                 AS operacion,
    A.tipo_operacion                                            AS tipo_operacion,
    A.sistema                                                   AS sistema,
    A.rut_cliente                                               AS rut_cliente,
    A.dv_rut_cliente                                            AS dv_rut_cliente ,
    "flag_existe_cliente_lir "                                  AS nombre_campo,
    cast(A.flag_existe_cliente_lir as string)                   AS valor_campo,
    "Cliente no informado como LIR"                             AS condicion_regla,
    "[eq to 0 (NOEXISTE)]"                                      AS valor_regla,           
    CASE 
         WHEN IFNULL(A.flag_existe_cliente_lir,0) = 0 
         THEN 1 
         ELSE 0 
    END                                                         AS flag_resultado_regla,    
    'S07'                                                       AS cod_evaluacion  
FROM 
    {base_silver_x}.tbl_cd_ope_condicion_salida_deterioro  A
"""


In [0]:
sql_safe(paso_query135)

sql_safe: query -> 
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CAMPO_EVAL_7 as
SELECT
    A.periodo_cierre                                            AS periodo_cierre,
    A.fecha_cierre                                              AS fecha_cierre,
    A.tipo_proceso                                              AS tipo_proceso,
    A.segmento                                                  AS segmento,
    A.operacion                                                 AS operacion,
    A.tipo_operacion                                            AS tipo_operacion,
    A.sistema                                                   AS sistema,
    A.rut_cliente                                               AS rut_cliente,
    A.dv_rut_cliente                                            AS dv_rut_cliente ,
    "flag_existe_cliente_lir "                                  AS nombre_campo,
    cast(A.flag_existe_cliente_lir as string)                   AS valor_campo,
    "Cliente no informad

DataFrame[]

### Evaluacion: Antiguedad clientes LIR (67)
---
* Si cliente tiene mas de 12 meses de antiguedad desde la ultima vez que se informo como lir entonces  flag =1 


In [0]:
paso_query140 =  f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CAMPO_EVAL_15 as
SELECT
    A.periodo_cierre                                            AS periodo_cierre,
    A.fecha_cierre                                              AS fecha_cierre,
    A.tipo_proceso                                              AS tipo_proceso,
    A.segmento                                                  AS segmento,
    A.operacion                                                 AS operacion,
    A.tipo_operacion                                            AS tipo_operacion,
    A.sistema                                                   AS sistema,
    A.rut_cliente                                               AS rut_cliente,
    A.dv_rut_cliente                                            AS dv_rut_cliente ,
    'cli_fecha_informada_lir'                                   AS nombre_campo,
    cast(IFNULL(A.cli_fecha_informada_lir,00000000) as string)  AS valor_campo,
    "Cliente informado como lir con antiguedad mayor a 12 meses"     AS condicion_regla,     
    "[ <= {p_fec_12_meses_atras}]"                                AS valor_regla,           
    CASE 
       WHEN IFNULL(A.cli_fecha_informada_lir,19000101) <= {p_fec_12_meses_atras}
       THEN 1 
       ELSE 0 
    END AS flag_resultado_regla,
    'S15'                                                       AS cod_evaluacion  
FROM 
    {base_silver_x}.tbl_cd_ope_condicion_salida_deterioro   A
"""


In [0]:
sql_safe(paso_query140)

sql_safe: query -> 
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CAMPO_EVAL_15 as
SELECT
    A.periodo_cierre                                            AS periodo_cierre,
    A.fecha_cierre                                              AS fecha_cierre,
    A.tipo_proceso                                              AS tipo_proceso,
    A.segmento                                                  AS segmento,
    A.operacion                                                 AS operacion,
    A.tipo_operacion                                            AS tipo_operacion,
    A.sistema                                                   AS sistema,
    A.rut_cliente                                               AS rut_cliente,
    A.dv_rut_cliente                                            AS dv_rut_cliente ,
    'cli_fecha_informada_lir'                                   AS nombre_campo,
    cast(IFNULL(A.cli_fecha_informada_lir,00000000) as string)  AS valor_campo,
    "Cliente informado 

DataFrame[]

### Evaluacion: Clientes con morosidad bci (60)
---
- Si cliente tiene mora menor a una cantidad parametrica ({p1_diasmora}) en periodo actual, entonces flag =1 


In [0]:
paso_query150 =  f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CAMPO_EVAL_8 as
SELECT
    A.periodo_cierre                                            AS periodo_cierre,
    A.fecha_cierre                                              AS fecha_cierre,
    A.tipo_proceso                                              AS tipo_proceso,
    A.segmento                                                  AS segmento,
    A.operacion                                                 AS operacion,
    A.tipo_operacion                                            AS tipo_operacion,
    A.sistema                                                   AS sistema,
    A.rut_cliente                                               AS rut_cliente,
    A.dv_rut_cliente                                            AS dv_rut_cliente ,
    'max_dia_mora_pact'                                          AS nombre_campo,
    cast(IFNULL(A.max_dia_mora_pact,0) as string)               AS valor_campo,
    "Cliente con maximo dias de mora menor o igual a"               AS condicion_regla,     
    "[<= {p1_diasmora}]"                                             AS valor_regla,           
    CASE 
       WHEN IFNULL(A.max_dia_mora_pact,0) <= {p1_diasmora}
       THEN 1 
       ELSE 0 
    END                                                         AS flag_resultado_regla,
    'S08'                                                       AS cod_evaluacion  
FROM 
    {base_silver_x}.tbl_cd_ope_condicion_salida_deterioro  A
"""

In [0]:
sql_safe(paso_query150)

sql_safe: query -> 
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CAMPO_EVAL_8 as
SELECT
    A.periodo_cierre                                            AS periodo_cierre,
    A.fecha_cierre                                              AS fecha_cierre,
    A.tipo_proceso                                              AS tipo_proceso,
    A.segmento                                                  AS segmento,
    A.operacion                                                 AS operacion,
    A.tipo_operacion                                            AS tipo_operacion,
    A.sistema                                                   AS sistema,
    A.rut_cliente                                               AS rut_cliente,
    A.dv_rut_cliente                                            AS dv_rut_cliente ,
    'max_dia_mora_pact'                                          AS nombre_campo,
    cast(IFNULL(A.max_dia_mora_pact,0) as string)               AS valor_campo,
    "Cliente con maximo

DataFrame[]

### Evaluacion: Operacion con 4 o mas pagos consecutivos (61)
---
- Si operacion tiene mas de n o mas pagos consecutivos entonces   flag =1 


In [0]:
paso_query160 =  f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CAMPO_EVAL_9 as
SELECT
    A.periodo_cierre                                            AS periodo_cierre,
    A.fecha_cierre                                              AS fecha_cierre,
    A.tipo_proceso                                              AS tipo_proceso,
    A.segmento                                                  AS segmento,
    A.operacion                                                 AS operacion,
    A.tipo_operacion                                            AS tipo_operacion,
    A.sistema                                                   AS sistema,
    A.rut_cliente                                               AS rut_cliente,
    A.dv_rut_cliente                                            AS dv_rut_cliente ,
    'cont_pag_cons'                                             AS nombre_campo,
    cast(IFNULL(A.cont_pag_cons,0) as string)                   AS valor_campo,
    "Operacion con pagos consecutivos mayor o igual a"          AS condicion_regla,     
    "[>={p_pagcons}]"                                           AS valor_regla,           
    CASE 
       WHEN IFNULL(A.cont_pag_cons,0) >= {p_pagcons}
       THEN 1 
       ELSE 0 
    END                                                         AS flag_resultado_regla,
    'S09'                                                       AS cod_evaluacion  
FROM 
    {base_silver_x}.tbl_cd_ope_condicion_salida_deterioro   A
"""

In [0]:
sql_safe(paso_query160)

sql_safe: query -> 
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CAMPO_EVAL_9 as
SELECT
    A.periodo_cierre                                            AS periodo_cierre,
    A.fecha_cierre                                              AS fecha_cierre,
    A.tipo_proceso                                              AS tipo_proceso,
    A.segmento                                                  AS segmento,
    A.operacion                                                 AS operacion,
    A.tipo_operacion                                            AS tipo_operacion,
    A.sistema                                                   AS sistema,
    A.rut_cliente                                               AS rut_cliente,
    A.dv_rut_cliente                                            AS dv_rut_cliente ,
    'cont_pag_cons'                                             AS nombre_campo,
    cast(IFNULL(A.cont_pag_cons,0) as string)                   AS valor_campo,
    "Operacion con pagos

DataFrame[]

### Evaluacion: Operacion con pagos parciales (63)
---
- Si operacion tiene mas de n o mas pagos parciales entonces   flag =1 

In [0]:
paso_query170 =  f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CAMPO_EVAL_10 as
SELECT
    A.periodo_cierre                                            AS periodo_cierre,
    A.fecha_cierre                                              AS fecha_cierre,
    A.tipo_proceso                                              AS tipo_proceso,
    A.segmento                                                  AS segmento,
    A.operacion                                                 AS operacion,
    A.tipo_operacion                                            AS tipo_operacion,
    A.sistema                                                   AS sistema,
    A.rut_cliente                                               AS rut_cliente,
    A.dv_rut_cliente                                            AS dv_rut_cliente ,
    'ope_pag_parciales_ini_cd'                                  AS nombre_campo,
    cast(IFNULL(A.ope_pag_parciales_ini_cd,0) as string)        AS valor_campo,
    "Operacion con pagos parciales mayor o igual a"             AS condicion_regla,     
    "[>= {p_cant_min_pag}]"                                     AS valor_regla,           
    CASE 
       WHEN IFNULL(A.ope_pag_parciales_ini_cd,0) >= {p_cant_min_pag}
       THEN 1 
       ELSE 0 
    END                                                         AS flag_resultado_regla,
    'S10'                                                       AS cod_evaluacion  
FROM 
    {base_silver_x}.tbl_cd_ope_condicion_salida_deterioro   A
"""

In [0]:
sql_safe(paso_query170)

sql_safe: query -> 
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CAMPO_EVAL_10 as
SELECT
    A.periodo_cierre                                            AS periodo_cierre,
    A.fecha_cierre                                              AS fecha_cierre,
    A.tipo_proceso                                              AS tipo_proceso,
    A.segmento                                                  AS segmento,
    A.operacion                                                 AS operacion,
    A.tipo_operacion                                            AS tipo_operacion,
    A.sistema                                                   AS sistema,
    A.rut_cliente                                               AS rut_cliente,
    A.dv_rut_cliente                                            AS dv_rut_cliente ,
    'ope_pag_parciales_ini_cd'                                  AS nombre_campo,
    cast(IFNULL(A.ope_pag_parciales_ini_cd,0) as string)        AS valor_campo,
    "Operacion con pago

DataFrame[]

### Evaluacion: Cliente operacion sin refinanciamiento ni curse bajo mora (64)
---
- Si cliente o operacion no tiene refinanciamiento o curse bajo mora entonces   flag =1 

In [0]:
paso_query190 =  f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CAMPO_EVAL_11 as
SELECT
    A.periodo_cierre                                            AS periodo_cierre,
    A.fecha_cierre                                              AS fecha_cierre,
    A.tipo_proceso                                              AS tipo_proceso,
    A.segmento                                                  AS segmento,
    A.operacion                                                 AS operacion,
    A.tipo_operacion                                            AS tipo_operacion,
    A.sistema                                                   AS sistema,
    A.rut_cliente                                               AS rut_cliente,
    A.dv_rut_cliente                                            AS dv_rut_cliente ,
    "flag_ope_ren_pact-flag_existe_operacion_rfz"                                  AS nombre_campo,
    concat('[',cast(A.flag_ope_ren_pact as string),'-', cast(A.flag_existe_operacion_rfz as string),']')                  AS valor_campo,
    "Cliente sin renegociados ni reestructuracio forzosa"                             AS condicion_regla,
    concat('[',"eq to 0 (NOREN)",' AND ',"eq to 0 (NORFZS)",']') AS valor_regla,           
    CASE 
         WHEN IFNULL(A.flag_ope_ren_pact,0) = 0 AND IFNULL(A.flag_existe_operacion_rfz,0) = 0
         THEN 1 
         ELSE 0 
    END                                                         AS flag_resultado_regla,    
    'S11'                                                       AS cod_evaluacion  
FROM 
    {base_silver_x}.tbl_cd_ope_condicion_salida_deterioro   A

"""

In [0]:
sql_safe(paso_query190)

sql_safe: query -> 
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CAMPO_EVAL_11 as
SELECT
    A.periodo_cierre                                            AS periodo_cierre,
    A.fecha_cierre                                              AS fecha_cierre,
    A.tipo_proceso                                              AS tipo_proceso,
    A.segmento                                                  AS segmento,
    A.operacion                                                 AS operacion,
    A.tipo_operacion                                            AS tipo_operacion,
    A.sistema                                                   AS sistema,
    A.rut_cliente                                               AS rut_cliente,
    A.dv_rut_cliente                                            AS dv_rut_cliente ,
    "flag_ope_ren_pact-flag_existe_operacion_rfz"                                  AS nombre_campo,
    concat('[',cast(A.flag_ope_ren_pact as string),'-', cast(A.flag_existe_operacion

DataFrame[]

### Evaluacion: Operaciones con pago de capital (65)
---
- Si operacion tiene pago de capital en mes actual entonces   flag =1 
- evalua si ha pagado capital desde que entro a deterioro la operacion
- el calculo para el campo ope_pag_capital_ifrs_ini_cd corresponde a la diferencia de capital desde la fecha de entrada a deterioro a la fecha actual de proceso


In [0]:
paso_query200 =  f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CAMPO_EVAL_12 as
SELECT
    A.periodo_cierre                                            AS periodo_cierre,
    A.fecha_cierre                                              AS fecha_cierre,
    A.tipo_proceso                                              AS tipo_proceso,
    A.segmento                                                  AS segmento,
    A.operacion                                                 AS operacion,
    A.tipo_operacion                                            AS tipo_operacion,
    A.sistema                                                   AS sistema,
    A.rut_cliente                                               AS rut_cliente,
    A.dv_rut_cliente                                            AS dv_rut_cliente ,
    'ope_pag_capital_ifrs_ini_cd'                                        AS nombre_campo,
    IFNULL(A.ope_pag_capital_ifrs_ini_cd,0)                              AS valor_campo,
    "Operacion con pago de saldo de capital ifrs"                    AS condicion_regla,     
    "[ >= {p_crit_det_sal_tot_ifrs } ]"                           AS valor_regla,           
    CASE 
       WHEN  IFNULL(A.ope_pag_capital_ifrs_ini_cd,0) > {p_crit_det_sal_tot_ifrs }
       THEN 1 
       ELSE 0 
    END                                                         AS flag_resultado_regla,
    'S12'                                                       AS cod_evaluacion  
FROM 
     {base_silver_x}.tbl_cd_ope_condicion_salida_deterioro   A
"""

In [0]:
sql_safe(paso_query200)

sql_safe: query -> 
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CAMPO_EVAL_12 as
SELECT
    A.periodo_cierre                                            AS periodo_cierre,
    A.fecha_cierre                                              AS fecha_cierre,
    A.tipo_proceso                                              AS tipo_proceso,
    A.segmento                                                  AS segmento,
    A.operacion                                                 AS operacion,
    A.tipo_operacion                                            AS tipo_operacion,
    A.sistema                                                   AS sistema,
    A.rut_cliente                                               AS rut_cliente,
    A.dv_rut_cliente                                            AS dv_rut_cliente ,
    'ope_pag_capital_ifrs_ini_cd'                                        AS nombre_campo,
    IFNULL(A.ope_pag_capital_ifrs_ini_cd,0)                              AS valor_campo,
    "

DataFrame[]

### Evaluacion: Operacion Minimo de meses en cartdet (66)
---
- Si cliente tiene un minimo de meses, {p_cantminmes},  en cartera deteriorada,  entonces   flag =1 

In [0]:
paso_query220 =  f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CAMPO_EVAL_13 as
SELECT
    A.periodo_cierre                                            AS periodo_cierre,
    A.fecha_cierre                                              AS fecha_cierre,
    A.tipo_proceso                                              AS tipo_proceso,
    A.segmento                                                  AS segmento,
    A.operacion                                                 AS operacion,
    A.tipo_operacion                                            AS tipo_operacion,
    A.sistema                                                   AS sistema,
    A.rut_cliente                                               AS rut_cliente,
    A.dv_rut_cliente                                            AS dv_rut_cliente ,
    'ope_num_meses_en_cd'                                                 AS nombre_campo,
    IFNULL(A.ope_num_meses_en_cd,0)                                       AS valor_campo,
    "Operacion con minimo de meses en cartera deteriorada mayor o igual a"      AS condicion_regla,     
    "[ >= {p_cantminmes} ]"                                              AS valor_regla,           
    CASE 
       WHEN IFNULL(A.ope_num_meses_en_cd,0) >= {p_cantminmes}
       THEN 1 
       ELSE 0 
    END                                                         AS flag_resultado_regla,
    'S13'                                                       AS cod_evaluacion  
FROM 
    {base_silver_x}.tbl_cd_ope_condicion_salida_deterioro   A
"""

In [0]:
sql_safe(paso_query220)

sql_safe: query -> 
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CAMPO_EVAL_13 as
SELECT
    A.periodo_cierre                                            AS periodo_cierre,
    A.fecha_cierre                                              AS fecha_cierre,
    A.tipo_proceso                                              AS tipo_proceso,
    A.segmento                                                  AS segmento,
    A.operacion                                                 AS operacion,
    A.tipo_operacion                                            AS tipo_operacion,
    A.sistema                                                   AS sistema,
    A.rut_cliente                                               AS rut_cliente,
    A.dv_rut_cliente                                            AS dv_rut_cliente ,
    'ope_num_meses_en_cd'                                                 AS nombre_campo,
    IFNULL(A.ope_num_meses_en_cd,0)                                       AS valor_campo,
   

DataFrame[]

### Evaluacion: Operaciones deterioradas periodo anterior no informadas en periodo actual (30)
---
- Si operacion está deteriorada en periodo anterior y no existe en periodo actual,  entonces   flag =1 

In [0]:
paso_query225 =  f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CAMPO_EVAL_14 as
SELECT
    A.periodo_cierre                                            AS periodo_cierre,
    A.fecha_cierre                                              AS fecha_cierre,
    A.tipo_proceso                                              AS tipo_proceso,
    A.segmento                                                  AS segmento,
    A.operacion                                                 AS operacion,
    A.tipo_operacion                                            AS tipo_operacion,
    A.sistema                                                   AS sistema,
    A.rut_cliente                                               AS rut_cliente,
    A.dv_rut_cliente                                            AS dv_rut_cliente ,
    'operacion - sistema'                                       AS nombre_campo,
    concat('[',CAST(IFNULL(B.operacion,'NOEXISTE-PACT') AS string),'-',CAST(IFNULL(B.sistema,'') as string),']')          AS valor_campo,
    "Operacion deteriorada periodo anterior no existe en periodo actual"    AS condicion_regla,     
    "[NOEXISTE-PACT]"          AS valor_regla,           
    CASE 
       WHEN trim(B.operacion) is null
       THEN 1 
       ELSE 0 
    END                                                         AS flag_resultado_regla,
    'S14'                                                       AS cod_evaluacion  
FROM 
    tmp_EXT_tbl_cartdet_crit_ent_crit A
LEFT JOIN
    dsr_gld_bciwork_db.tbl_cd_ope_condicion_salida_deterioro   B
ON  trim(A.operacion) = trim(B.operacion)
    AND trim(A.sistema) = trim(B.sistema)
"""

In [0]:
sql_safe(paso_query225)

sql_safe: query -> 
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CAMPO_EVAL_14 as
SELECT
    A.periodo_cierre                                            AS periodo_cierre,
    A.fecha_cierre                                              AS fecha_cierre,
    A.tipo_proceso                                              AS tipo_proceso,
    A.segmento                                                  AS segmento,
    A.operacion                                                 AS operacion,
    A.tipo_operacion                                            AS tipo_operacion,
    A.sistema                                                   AS sistema,
    A.rut_cliente                                               AS rut_cliente,
    A.dv_rut_cliente                                            AS dv_rut_cliente ,
    'operacion - sistema'                                       AS nombre_campo,
    concat('[',CAST(IFNULL(B.operacion,'NOEXISTE-PACT') AS string),'-',CAST(IFNULL(B.sistema,'') as str

DataFrame[]

### Salida Temporal a Nivel de Campo Evaludado (tmp_tbl_cartdet_crit_ent_ope_campo)
------------------
* generar salida temporal a nivel de campo evaluado. 
* se registran todas las operaciones evaluadas


In [0]:

paso_query250 = f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_tbl_cartdet_crit_sal_ope_eval AS
SELECT * FROM   tmp_RES_D00_OPE_CAMPO_EVAL_1 
UNION 
SELECT * FROM   tmp_RES_D00_OPE_CAMPO_EVAL_2 
UNION
SELECT * FROM   tmp_RES_D00_OPE_CAMPO_EVAL_3 
UNION
SELECT * FROM   tmp_RES_D00_OPE_CAMPO_EVAL_4
UNION
SELECT * FROM   tmp_RES_D00_OPE_CAMPO_EVAL_5
UNION
SELECT * FROM   tmp_RES_D00_OPE_CAMPO_EVAL_6
UNION
SELECT * FROM   tmp_RES_D00_OPE_CAMPO_EVAL_7
UNION
SELECT * FROM   tmp_RES_D00_OPE_CAMPO_EVAL_8
UNION
SELECT * FROM   tmp_RES_D00_OPE_CAMPO_EVAL_9
UNION
SELECT * FROM   tmp_RES_D00_OPE_CAMPO_EVAL_10
UNION
SELECT * FROM   tmp_RES_D00_OPE_CAMPO_EVAL_11
UNION
SELECT * FROM   tmp_RES_D00_OPE_CAMPO_EVAL_12
UNION
SELECT * FROM   tmp_RES_D00_OPE_CAMPO_EVAL_13
UNION
SELECT * FROM   tmp_RES_D00_OPE_CAMPO_EVAL_14
UNION
SELECT * FROM   tmp_RES_D00_OPE_CAMPO_EVAL_15
"""  

In [0]:
sql_safe(paso_query250)

sql_safe: query -> 
CREATE OR REPLACE TEMPORARY VIEW tmp_tbl_cartdet_crit_sal_ope_eval AS
SELECT * FROM   tmp_RES_D00_OPE_CAMPO_EVAL_1 
UNION 
SELECT * FROM   tmp_RES_D00_OPE_CAMPO_EVAL_2 
UNION
SELECT * FROM   tmp_RES_D00_OPE_CAMPO_EVAL_3 
UNION
SELECT * FROM   tmp_RES_D00_OPE_CAMPO_EVAL_4
UNION
SELECT * FROM   tmp_RES_D00_OPE_CAMPO_EVAL_5
UNION
SELECT * FROM   tmp_RES_D00_OPE_CAMPO_EVAL_6
UNION
SELECT * FROM   tmp_RES_D00_OPE_CAMPO_EVAL_7
UNION
SELECT * FROM   tmp_RES_D00_OPE_CAMPO_EVAL_8
UNION
SELECT * FROM   tmp_RES_D00_OPE_CAMPO_EVAL_9
UNION
SELECT * FROM   tmp_RES_D00_OPE_CAMPO_EVAL_10
UNION
SELECT * FROM   tmp_RES_D00_OPE_CAMPO_EVAL_11
UNION
SELECT * FROM   tmp_RES_D00_OPE_CAMPO_EVAL_12
UNION
SELECT * FROM   tmp_RES_D00_OPE_CAMPO_EVAL_13
UNION
SELECT * FROM   tmp_RES_D00_OPE_CAMPO_EVAL_14
UNION
SELECT * FROM   tmp_RES_D00_OPE_CAMPO_EVAL_15



DataFrame[]

## Carga Tablas de Salidas
--------------------------------------
* carga resultados a tablas de salidas del notebook

### Carga Tabla Evaluacion 


#### Reproceso (Elimina registros en caso de reprocesos)

In [0]:
paso_query300 = f""" TRUNCATE TABLE {base_silver_x}.tbl_cd_cartdet_crit_sal_ope_eval """

In [0]:
sql_safe(paso_query300)

sql_safe: query ->  TRUNCATE TABLE dsr_gld_bciwork_db.tbl_cd_cartdet_crit_sal_ope_eval 


DataFrame[]

#### Inserta Registros tabla salida

In [0]:

paso_query310 = f"""
INSERT INTO {base_silver_x}.tbl_cd_cartdet_crit_sal_ope_eval
SELECT 
    IFNULL(periodo_cierre,19000101),
    IFNULL(fecha_cierre,190001),
    IFNULL(tipo_proceso,' '),
    IFNULL(segmento,' '),
    IFNULL(operacion,' '),
    IFNULL(tipo_operacion,' '),
    IFNULL(sistema,' '),
    IFNULL(rut_cliente,0),
    IFNULL(dv_rut_cliente,' '),
    IFNULL(nombre_campo,' '),
    IFNULL(valor_campo,' '),
    IFNULL(condicion_regla,' '),
    IFNULL(valor_regla,' '),
    IFNULL(flag_resultado_regla,0),
    IFNULL(cod_evaluacion,' ')
FROM
  tmp_tbl_cartdet_crit_sal_ope_eval
"""  


In [0]:
sql_safe(paso_query310)

sql_safe: query -> 
INSERT INTO dsr_gld_bciwork_db.tbl_cd_cartdet_crit_sal_ope_eval
SELECT 
    IFNULL(periodo_cierre,19000101),
    IFNULL(fecha_cierre,190001),
    IFNULL(tipo_proceso,' '),
    IFNULL(segmento,' '),
    IFNULL(operacion,' '),
    IFNULL(tipo_operacion,' '),
    IFNULL(sistema,' '),
    IFNULL(rut_cliente,0),
    IFNULL(dv_rut_cliente,' '),
    IFNULL(nombre_campo,' '),
    IFNULL(valor_campo,' '),
    IFNULL(condicion_regla,' '),
    IFNULL(valor_regla,' '),
    IFNULL(flag_resultado_regla,0),
    IFNULL(cod_evaluacion,' ')
FROM
  tmp_tbl_cartdet_crit_sal_ope_eval



DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

##ESTADISTICAS

In [0]:
%sql
select 
fecha_cierre, 
cod_evaluacion, 
count(1) 
from  ${bci.dbnamesilver}.tbl_cd_cartdet_crit_sal_ope_eval 
group by 1,2 order by 1,2

fecha_cierre,cod_evaluacion,count(1)
20250930,S01,294071
20250930,S02,294071
20250930,S03,294071
20250930,S04,294071
20250930,S05,294071
20250930,S06,294071
20250930,S07,294071
20250930,S08,294071
20250930,S09,294071
20250930,S10,294071


## Mensaje termino OK

In [0]:
msgerrorx="OK"
dbutils.notebook.exit("{\"coderror\":0, \"msgerror\":\""+msgerrorx+"\"}")